# Data Modelling & Visualisation — Part 1: Modelling Design and Training

**INFO 442 · Data Science Projects · M5**  
**Project:** Gaming Behavior & Academic Performance

---

## Division of work

This notebook is designed to be shared by two teammates.

### Part 1 — completed in this section

This part focuses on **modelling rationale and model construction**, not final result storytelling.

It includes:

1. M1–M4 continuity and modelling question
2. Data loading from the M3 processed dataset
3. Modelling data audit
4. Feature group audit
5. EDA-informed feature engineering
6. Leakage-risk feature decision
7. Multiple modelling design matrices
8. Baseline, linear, threshold, regularized, and tree-based model training
9. 5-fold cross-validation on training data
10. Held-out test predictions for Part 2
11. Coefficient and raw feature-importance exports
12. Optional tuning hooks for the next teammate if needed

### Part 2 — to be completed later

The next teammate should use the outputs from this notebook to complete:

1. Final model evaluation interpretation
2. Performance comparison table/chart
3. Stakeholder-facing visualisations
4. Residual analysis
5. Model limitations and potential failure modes

---

## Modelling question

Based on M1–M4, the M5 modelling question is:

> Can student grades be predicted from gaming behaviour, study habits, and health-related variables, and do nonlinear or threshold-aware models better capture the gaming-risk pattern identified during EDA?

The target variable is `grades`, so this is a **regression problem**.

In [22]:
# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

RANDOM_STATE = 42

# Try several likely folders so the notebook can run in different repo layouts.
FALLBACK_DIRS = [
    '../../data/processed/',
    '../data/processed/',
    'data/processed/',
    './',
    '/mnt/data/'
]

OUTPUT_DIR = 'm5_outputs_part1/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR + 'tables/', exist_ok=True)
os.makedirs(OUTPUT_DIR + 'models/', exist_ok=True)

## 1. Load M3 processed data

M5 does not redo cleaning.  
It uses the train/test files produced by M3 so that the experimental design remains consistent.

Expected files:

- `X_train.csv`
- `X_test.csv`
- `X_train_scaled.csv`
- `X_test_scaled.csv`
- `y_train.csv`
- `y_test.csv`
- `scaler.pkl`

In [23]:
def find_processed_dir(fallback_dirs):
    required = [
        'X_train.csv', 'X_test.csv',
        'X_train_scaled.csv', 'X_test_scaled.csv',
        'y_train.csv', 'y_test.csv'
    ]
    for folder in fallback_dirs:
        if all(os.path.exists(os.path.join(folder, f)) for f in required):
            return folder
    raise FileNotFoundError('Processed files not found. Please check the data path.')


def read_target(path):
    y_df = pd.read_csv(path)
    y_df = y_df.drop(columns=[c for c in y_df.columns if c.startswith('Unnamed')], errors='ignore')

    if 'grades' in y_df.columns:
        return y_df['grades'].astype(float)
    if y_df.shape[1] == 1:
        return y_df.iloc[:, 0].astype(float)

    raise ValueError(f'Cannot identify target column in {path}. Columns: {list(y_df.columns)}')


def clean_feature_frame(df):
    df = df.copy()
    df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')], errors='ignore')
    df = df.drop(columns=['student_id'], errors='ignore')

    for col in df.select_dtypes(include=['bool']).columns:
        df[col] = df[col].astype(int)

    return df


DATA_PROCESSED_DIR = find_processed_dir(FALLBACK_DIRS)
print(f'Using processed data directory: {DATA_PROCESSED_DIR}')

X_train_raw = clean_feature_frame(pd.read_csv(DATA_PROCESSED_DIR + 'X_train.csv'))
X_test_raw = clean_feature_frame(pd.read_csv(DATA_PROCESSED_DIR + 'X_test.csv'))

X_train_scaled_m3 = clean_feature_frame(pd.read_csv(DATA_PROCESSED_DIR + 'X_train_scaled.csv'))
X_test_scaled_m3 = clean_feature_frame(pd.read_csv(DATA_PROCESSED_DIR + 'X_test_scaled.csv'))

y_train = read_target(DATA_PROCESSED_DIR + 'y_train.csv')
y_test = read_target(DATA_PROCESSED_DIR + 'y_test.csv')

# Align columns.
X_train_raw = X_train_raw[[c for c in X_train_raw.columns if c in X_test_raw.columns]]
X_test_raw = X_test_raw[X_train_raw.columns]

X_train_scaled_m3 = X_train_scaled_m3[[c for c in X_train_scaled_m3.columns if c in X_test_scaled_m3.columns]]
X_test_scaled_m3 = X_test_scaled_m3[X_train_scaled_m3.columns]

print(f'X_train_raw: {X_train_raw.shape}')
print(f'X_test_raw: {X_test_raw.shape}')
print(f'X_train_scaled_m3: {X_train_scaled_m3.shape}')
print(f'X_test_scaled_m3: {X_test_scaled_m3.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train_raw.head()

Using processed data directory: ../../data/processed/
X_train_raw: (6400, 20)
X_test_raw: (1600, 20)
X_train_scaled_m3: (6400, 20)
X_test_scaled_m3: (1600, 20)
y_train: (6400,)
y_test: (1600,)


,age,gaming_hours,study_hours,sleep_hours,attendance,social_activity,device_usage,reaction_time_ms,addiction_score,gender_Male,gender_Other,genre_Casual,genre_FPS,genre_RPG,stress_level_ord,gaming_hours_sq,gaming_hours_x_FPS,gaming_hours_x_RPG,gaming_study_ratio,total_load
0,22,2.31,4.46,7.81,98.29,3.68,7.60,288.18,4.17,0,0,1,0,0,1,5.3361,0.00,0.0,0.516779,6.77
1,21,4.99,3.45,5.82,90.59,3.01,9.93,254.85,12.71,1,0,0,1,0,1,24.9001,4.99,0.0,1.442197,8.44
2,21,7.14,7.64,8.97,87.22,0.30,9.37,225.75,14.05,0,0,0,1,0,0,50.9796,7.14,0.0,0.933333,14.78
3,19,3.52,9.59,7.16,84.28,4.61,5.31,276.24,7.44,0,0,1,0,0,1,12.3904,0.00,0.0,0.366667,13.11
4,16,0.76,1.52,6.42,70.51,1.17,5.98,310.51,6.39,1,0,0,1,0,1,0.5776,0.76,0.0,0.496732,2.28


## 2. Modelling data audit

Before fitting models, we document whether the processed data looks usable for modelling.

This is different from M3 cleaning.  
Here the goal is to check that the files passed from preprocessing into modelling have the expected structure.

In [24]:
audit = {}

audit['n_train_rows'] = X_train_raw.shape[0]
audit['n_test_rows'] = X_test_raw.shape[0]
audit['n_features_raw'] = X_train_raw.shape[1]
audit['n_features_scaled'] = X_train_scaled_m3.shape[1]
audit['train_target_mean'] = y_train.mean()
audit['test_target_mean'] = y_test.mean()
audit['train_target_std'] = y_train.std()
audit['test_target_std'] = y_test.std()
audit['column_alignment_raw'] = list(X_train_raw.columns) == list(X_test_raw.columns)
audit['column_alignment_scaled'] = list(X_train_scaled_m3.columns) == list(X_test_scaled_m3.columns)
audit['missing_values_X_train'] = int(X_train_raw.isna().sum().sum())
audit['missing_values_X_test'] = int(X_test_raw.isna().sum().sum())
audit['missing_values_y_train'] = int(y_train.isna().sum())
audit['missing_values_y_test'] = int(y_test.isna().sum())

audit_df = pd.DataFrame([audit])
audit_df.to_csv(OUTPUT_DIR + 'tables/modelling_data_audit.csv', index=False)

audit_df

,n_train_rows,n_test_rows,n_features_raw,n_features_scaled,train_target_mean,test_target_mean,train_target_std,test_target_std,column_alignment_raw,column_alignment_scaled,missing_values_X_train,missing_values_X_test,missing_values_y_train,missing_values_y_test
0,6400,1600,20,20,66.025109,66.803444,22.511218,22.057604,True,True,0,0,0,0


**Interpretation:**  
This table is a handoff check. It confirms that Part 1 is using the expected modelling data and gives Part 2 a record of the train/test setup.

## 3. Feature group audit

The feature audit connects variables to the project logic.

Feature groups:

- gaming behaviour
- study behaviour
- health / wellbeing
- demographic and encoded categorical variables
- engineered variables

The next teammate can use this table when interpreting feature importance.

In [25]:
feature_groups = {
    'gaming_behavior': [
        'gaming_hours',
        'gaming_hours_sq',
        'gaming_study_ratio',
        'gaming_hours_x_FPS',
        'gaming_hours_x_RPG'
    ],
    'study_behavior': [
        'study_hours',
        'attendance',
        'total_load'
    ],
    'health_wellbeing': [
        'sleep_hours',
        'stress_level_ord',
        'addiction_score',
        'social_activity',
        'device_usage',
        'reaction_time_ms'
    ],
    'demographic_or_categorical': [
        'age',
        'gender_Male',
        'gender_Other',
        'genre_Casual',
        'genre_FPS',
        'genre_RPG'
    ]
}

feature_audit_rows = []

for group, features in feature_groups.items():
    for feature in features:
        feature_audit_rows.append({
            'feature_group': group,
            'feature': feature,
            'exists_in_dataset': feature in X_train_raw.columns
        })

feature_audit = pd.DataFrame(feature_audit_rows)
feature_audit.to_csv(OUTPUT_DIR + 'tables/feature_audit.csv', index=False)

feature_audit

,feature_group,feature,exists_in_dataset
0,gaming_behavior,gaming_hours,True
1,gaming_behavior,gaming_hours_sq,True
2,gaming_behavior,gaming_study_ratio,True
3,gaming_behavior,gaming_hours_x_FPS,True
4,gaming_behavior,gaming_hours_x_RPG,True
5,study_behavior,study_hours,True
6,study_behavior,attendance,True
7,study_behavior,total_load,True
8,health_wellbeing,sleep_hours,True
9,health_wellbeing,stress_level_ord,True


## 4. EDA-informed feature engineering

M4 suggested a possible threshold around 2 gaming hours per day.  
To turn this finding into a model-testable feature, Part 1 creates piecewise features.

Main threshold features:

```text
gaming_within_2h = min(gaming_hours, 2)
gaming_over_2h = max(0, gaming_hours - 2)
```

Sensitivity thresholds are also created:

```text
gaming_over_1_5h
gaming_over_2_5h
```

These do not prove a causal threshold. They only allow the models to represent the EDA pattern.

In [26]:
def add_threshold_features(df):
    df = df.copy()

    if 'gaming_hours' in df.columns:
        df['gaming_within_2h'] = np.minimum(df['gaming_hours'], 2.0)
        df['gaming_over_2h'] = np.maximum(0, df['gaming_hours'] - 2.0)

        # Sensitivity features for nearby thresholds.
        df['gaming_over_1_5h'] = np.maximum(0, df['gaming_hours'] - 1.5)
        df['gaming_over_2_5h'] = np.maximum(0, df['gaming_hours'] - 2.5)

    return df


LEAKAGE_RISK_FEATURES = ['reaction_time_ms']

X_train_model = X_train_raw.drop(columns=LEAKAGE_RISK_FEATURES, errors='ignore')
X_test_model = X_test_raw.drop(columns=LEAKAGE_RISK_FEATURES, errors='ignore')

X_train_model = add_threshold_features(X_train_model)
X_test_model = add_threshold_features(X_test_model)

print(f'Features before modelling decisions: {X_train_raw.shape[1]}')
print(f'Features after leakage-risk exclusion and threshold engineering: {X_train_model.shape[1]}')
print('Excluded features:', LEAKAGE_RISK_FEATURES)

X_train_model.head()

Features before modelling decisions: 20
Features after leakage-risk exclusion and threshold engineering: 23
Excluded features: ['reaction_time_ms']


,age,gaming_hours,study_hours,sleep_hours,attendance,social_activity,device_usage,addiction_score,gender_Male,gender_Other,genre_Casual,genre_FPS,genre_RPG,stress_level_ord,gaming_hours_sq,gaming_hours_x_FPS,gaming_hours_x_RPG,gaming_study_ratio,total_load,gaming_within_2h,gaming_over_2h,gaming_over_1_5h,gaming_over_2_5h
0,22,2.31,4.46,7.81,98.29,3.68,7.60,4.17,0,0,1,0,0,1,5.3361,0.00,0.0,0.516779,6.77,2.00,0.31,0.81,0.00
1,21,4.99,3.45,5.82,90.59,3.01,9.93,12.71,1,0,0,1,0,1,24.9001,4.99,0.0,1.442197,8.44,2.00,2.99,3.49,2.49
2,21,7.14,7.64,8.97,87.22,0.30,9.37,14.05,0,0,0,1,0,0,50.9796,7.14,0.0,0.933333,14.78,2.00,5.14,5.64,4.64
3,19,3.52,9.59,7.16,84.28,4.61,5.31,7.44,0,0,1,0,0,1,12.3904,0.00,0.0,0.366667,13.11,2.00,1.52,2.02,1.02
4,16,0.76,1.52,6.42,70.51,1.17,5.98,6.39,1,0,0,1,0,1,0.5776,0.76,0.0,0.496732,2.28,0.76,0.00,0.00,0.00


## 5. Define modelling design matrices

Instead of using one single feature set, Part 1 prepares several design matrices.  
This makes the modelling work more substantial and gives Part 2 a richer comparison.

Design matrices:

1. **Core behavioural features** — small, interpretable feature set.
2. **Core + 2h threshold features** — directly tests the M4 threshold finding.
3. **Core + threshold sensitivity features** — checks nearby threshold choices.
4. **Full processed features** — all available processed variables excluding leakage-risk feature.
5. **Full processed features without device usage** — optional multicollinearity sensitivity check.

In [27]:
core_features = [
    'gaming_hours',
    'study_hours',
    'sleep_hours',
    'attendance',
    'addiction_score',
    'stress_level_ord'
]
core_features = [c for c in core_features if c in X_train_model.columns]

core_threshold_2h_features = core_features + [
    c for c in ['gaming_within_2h', 'gaming_over_2h'] if c in X_train_model.columns
]

core_threshold_sensitivity_features = core_features + [
    c for c in ['gaming_over_1_5h', 'gaming_over_2h', 'gaming_over_2_5h'] if c in X_train_model.columns
]

full_features = X_train_model.columns.tolist()

full_no_device_features = [c for c in full_features if c != 'device_usage']

design_matrices = {
    'core_features': core_features,
    'core_threshold_2h': core_threshold_2h_features,
    'core_threshold_sensitivity': core_threshold_sensitivity_features,
    'full_features': full_features,
    'full_no_device_usage': full_no_device_features
}

design_matrix_rows = []

for name, features in design_matrices.items():
    design_matrix_rows.append({
        'design_matrix': name,
        'n_features': len(features),
        'features': ', '.join(features)
    })

design_matrix_table = pd.DataFrame(design_matrix_rows)
design_matrix_table.to_csv(OUTPUT_DIR + 'tables/design_matrix_plan.csv', index=False)

design_matrix_table

,design_matrix,n_features,features
0,core_features,6,"gaming_hours, study_hours, sleep_hours, attend..."
1,core_threshold_2h,8,"gaming_hours, study_hours, sleep_hours, attend..."
2,core_threshold_sensitivity,9,"gaming_hours, study_hours, sleep_hours, attend..."
3,full_features,23,"age, gaming_hours, study_hours, sleep_hours, a..."
4,full_no_device_usage,22,"age, gaming_hours, study_hours, sleep_hours, a..."


**Interpretation:**  
The design matrix table shows how the modelling plan connects to EDA findings and data-quality risks.  
Part 2 can use this to explain whether added complexity improves performance.

## 6. Model selection rationale

The model sequence follows the Week 7 idea of building from sensible baselines to more flexible models.

Models included:

- Dummy baseline
- Linear core model
- Linear full model
- Piecewise / threshold linear model
- Threshold sensitivity linear model
- Ridge
- Lasso
- ElasticNet
- Random Forest
- Gradient Boosting

This gives the evaluation teammate enough material to compare both **model family** and **feature design**.

In [28]:
model_specs = []

# Baseline
model_specs.append({
    'Model': 'Dummy Mean Baseline',
    'Short_name': 'dummy_mean_baseline',
    'Model_object': DummyRegressor(strategy='mean'),
    'X_train': X_train_model[full_features],
    'X_test': X_test_model[full_features],
    'Feature_set': 'full_features',
    'Model_family': 'baseline'
})

# Linear models
linear_designs = [
    ('Linear Regression Core', 'linear_regression_core', 'core_features'),
    ('Linear Regression Full', 'linear_regression_full', 'full_features'),
    ('Piecewise Linear 2h', 'piecewise_linear_2h', 'core_threshold_2h'),
    ('Piecewise Linear Sensitivity', 'piecewise_linear_sensitivity', 'core_threshold_sensitivity')
]

for model_name, short_name, design_name in linear_designs:
    features = design_matrices[design_name]
    model_specs.append({
        'Model': model_name,
        'Short_name': short_name,
        'Model_object': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LinearRegression())
        ]),
        'X_train': X_train_model[features],
        'X_test': X_test_model[features],
        'Feature_set': design_name,
        'Model_family': 'linear'
    })

# Regularized models
regularized_models = [
    ('Ridge Regression', 'ridge_regression', RidgeCV(alphas=np.logspace(-3, 3, 25))),
    ('Lasso Regression', 'lasso_regression', LassoCV(
        alphas=np.logspace(-4, 1, 30),
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=20000
    )),
    ('ElasticNet Regression', 'elasticnet_regression', ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-4, 1, 30),
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=20000
    ))
]

for model_name, short_name, estimator in regularized_models:
    model_specs.append({
        'Model': model_name,
        'Short_name': short_name,
        'Model_object': Pipeline([
            ('scaler', StandardScaler()),
            ('model', estimator)
        ]),
        'X_train': X_train_model[full_features],
        'X_test': X_test_model[full_features],
        'Feature_set': 'full_features',
        'Model_family': 'regularized_linear'
    })

# Tree models
tree_models = [
    ('Random Forest', 'random_forest', RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
    ('Gradient Boosting', 'gradient_boosting', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        random_state=RANDOM_STATE
    ))
]

for model_name, short_name, estimator in tree_models:
    model_specs.append({
        'Model': model_name,
        'Short_name': short_name,
        'Model_object': estimator,
        'X_train': X_train_model[full_features],
        'X_test': X_test_model[full_features],
        'Feature_set': 'full_features',
        'Model_family': 'tree_ensemble'
    })

# Device-usage sensitivity models
model_specs.append({
    'Model': 'Ridge No Device Usage',
    'Short_name': 'ridge_no_device_usage',
    'Model_object': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RidgeCV(alphas=np.logspace(-3, 3, 25)))
    ]),
    'X_train': X_train_model[full_no_device_features],
    'X_test': X_test_model[full_no_device_features],
    'Feature_set': 'full_no_device_usage',
    'Model_family': 'regularized_linear_sensitivity'
})

model_training_plan = pd.DataFrame([
    {
        'Model': spec['Model'],
        'Short_name': spec['Short_name'],
        'Model_family': spec['Model_family'],
        'Feature_set': spec['Feature_set'],
        'Number_of_features': spec['X_train'].shape[1]
    }
    for spec in model_specs
])

model_training_plan.to_csv(OUTPUT_DIR + 'tables/model_training_plan.csv', index=False)

model_training_plan

,Model,Short_name,Model_family,Feature_set,Number_of_features
0,Dummy Mean Baseline,dummy_mean_baseline,baseline,full_features,23
1,Linear Regression Core,linear_regression_core,linear,core_features,6
2,Linear Regression Full,linear_regression_full,linear,full_features,23
3,Piecewise Linear 2h,piecewise_linear_2h,linear,core_threshold_2h,8
4,Piecewise Linear Sensitivity,piecewise_linear_sensitivity,linear,core_threshold_sensitivity,9
5,Ridge Regression,ridge_regression,regularized_linear,full_features,23
6,Lasso Regression,lasso_regression,regularized_linear,full_features,23
7,ElasticNet Regression,elasticnet_regression,regularized_linear,full_features,23
8,Random Forest,random_forest,tree_ensemble,full_features,23
9,Gradient Boosting,gradient_boosting,tree_ensemble,full_features,23


## 7. Evaluation metrics prepared for Part 2

Part 1 computes raw metrics but does not write final conclusions.

Metrics:

- **RMSE**: penalizes large errors.
- **MAE**: average grade-point error, easiest to explain to stakeholders.
- **R²**: variance explained.

5-fold cross-validation is computed on the training set.  
The held-out test set is used to generate prediction files for Part 2.

In [29]:
def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2


def run_cross_validation(model, X, y):
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    scoring = {
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error',
        'r2': 'r2'
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        'CV_RMSE_mean': -scores['test_rmse'].mean(),
        'CV_RMSE_std': scores['test_rmse'].std(),
        'CV_MAE_mean': -scores['test_mae'].mean(),
        'CV_MAE_std': scores['test_mae'].std(),
        'CV_R2_mean': scores['test_r2'].mean(),
        'CV_R2_std': scores['test_r2'].std()
    }

## 8. Train models and save outputs

This section trains all Part 1 models and saves outputs for the evaluation teammate.

In [30]:
performance_rows = []
metadata_rows = []
test_predictions = pd.DataFrame({'y_true': y_test.reset_index(drop=True)})

for spec in model_specs:
    model_name = spec['Model']
    short_name = spec['Short_name']
    model = spec['Model_object']
    Xtr = spec['X_train']
    Xte = spec['X_test']

    print(f'Training model: {model_name}')

    cv_result = run_cross_validation(model, Xtr, y_train)

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    test_rmse, test_mae, test_r2 = regression_metrics(y_test, y_pred)

    test_predictions[f'pred_{short_name}'] = y_pred

    performance_rows.append({
        'Model': model_name,
        'Short_name': short_name,
        'Model_family': spec['Model_family'],
        'Feature_set': spec['Feature_set'],
        'Number_of_features': Xtr.shape[1],
        **cv_result,
        'Test_RMSE': test_rmse,
        'Test_MAE': test_mae,
        'Test_R2': test_r2
    })

    metadata_rows.append({
        'Model': model_name,
        'Short_name': short_name,
        'Model_family': spec['Model_family'],
        'Feature_set': spec['Feature_set'],
        'Number_of_features': Xtr.shape[1],
        'Excluded_features': ', '.join(LEAKAGE_RISK_FEATURES),
        'Threshold_features_added': 'gaming_within_2h, gaming_over_2h, gaming_over_1_5h, gaming_over_2_5h'
    })

    joblib.dump(model, OUTPUT_DIR + f'models/{short_name}.joblib')

model_performance_raw = pd.DataFrame(performance_rows).sort_values('Test_RMSE').reset_index(drop=True)
model_metadata = pd.DataFrame(metadata_rows)

model_performance_raw.to_csv(OUTPUT_DIR + 'tables/model_performance_raw.csv', index=False)
test_predictions.to_csv(OUTPUT_DIR + 'tables/test_predictions.csv', index=False)
model_metadata.to_csv(OUTPUT_DIR + 'tables/trained_model_metadata.csv', index=False)

print('Training complete.')
print(f'Outputs saved to: {OUTPUT_DIR}')

model_performance_raw

Training model: Dummy Mean Baseline
Training model: Linear Regression Core
Training model: Linear Regression Full
Training model: Piecewise Linear 2h
Training model: Piecewise Linear Sensitivity
Training model: Ridge Regression
Training model: Lasso Regression
Training model: ElasticNet Regression
Training model: Random Forest
Training model: Gradient Boosting
Training model: Ridge No Device Usage
Training complete.
Outputs saved to: m5_outputs_part1/


,Model,Short_name,Model_family,Feature_set,Number_of_features,CV_RMSE_mean,CV_RMSE_std,CV_MAE_mean,CV_MAE_std,CV_R2_mean,CV_R2_std,Test_RMSE,Test_MAE,Test_R2
0,Gradient Boosting,gradient_boosting,tree_ensemble,full_features,23,5.780924,0.063273,4.553909,0.062443,0.933975,0.001072,5.598219,4.424224,0.935545
1,Random Forest,random_forest,tree_ensemble,full_features,23,5.941243,0.090605,4.656260,0.071911,0.930257,0.001740,5.725355,4.495107,0.932585
2,Lasso Regression,lasso_regression,regularized_linear,full_features,23,6.332476,0.034201,4.986150,0.043959,0.920778,0.000322,6.183781,4.964183,0.921356
3,ElasticNet Regression,elasticnet_regression,regularized_linear,full_features,23,6.333233,0.034216,4.986770,0.043816,0.920759,0.000317,6.183839,4.964305,0.921355
4,Linear Regression Full,linear_regression_full,linear,full_features,23,6.333179,0.034640,4.986508,0.044502,0.920760,0.000295,6.184174,4.965206,0.921346
5,Ridge Regression,ridge_regression,regularized_linear,full_features,23,6.333335,0.034512,4.986937,0.044092,0.920756,0.000315,6.184506,4.965227,0.921338
6,Ridge No Device Usage,ridge_no_device_usage,regularized_linear_sensitivity,full_no_device_usage,22,6.332516,0.033476,4.985975,0.043294,0.920777,0.000311,6.184645,4.965570,0.921334
7,Piecewise Linear Sensitivity,piecewise_linear_sensitivity,linear,core_threshold_sensitivity,9,6.459098,0.069476,5.080055,0.061721,0.917576,0.001238,6.239704,4.975529,0.919928
8,Piecewise Linear 2h,piecewise_linear_2h,linear,core_threshold_2h,8,6.509146,0.079539,5.133877,0.071522,0.916288,0.001772,6.333379,5.065223,0.917505
9,Linear Regression Core,linear_regression_core,linear,core_features,6,6.680767,0.100005,5.265843,0.072614,0.911802,0.002664,6.478026,5.170774,0.913694


**Note for Part 2:**  
This is a raw model-output table, not the final presentation table.  
The next teammate should clean it into a comparison table or chart and write the interpretation.

## 9. Export coefficients and feature importance for Part 2

This section prepares interpretation materials but does not create final stakeholder-facing plots.

In [31]:
coefficient_rows = []

for spec in model_specs:
    fitted_model = spec['Model_object']

    if isinstance(fitted_model, Pipeline):
        final_estimator = fitted_model.named_steps['model']
        feature_names = spec['X_train'].columns.tolist()

        if hasattr(final_estimator, 'coef_'):
            for feature, coef in zip(feature_names, final_estimator.coef_):
                coefficient_rows.append({
                    'Model': spec['Model'],
                    'Short_name': spec['Short_name'],
                    'Feature': feature,
                    'Coefficient': coef
                })

linear_coefficients = pd.DataFrame(coefficient_rows)

if len(linear_coefficients) > 0:
    linear_coefficients.to_csv(OUTPUT_DIR + 'tables/linear_model_coefficients.csv', index=False)
    display(linear_coefficients.head(20))
else:
    print('No linear coefficients found.')

,Model,Short_name,Feature,Coefficient
0,Linear Regression Core,linear_regression_core,gaming_hours,-11.683930
1,Linear Regression Core,linear_regression_core,study_hours,15.863516
2,Linear Regression Core,linear_regression_core,sleep_hours,5.768578
3,Linear Regression Core,linear_regression_core,attendance,3.207279
4,Linear Regression Core,linear_regression_core,addiction_score,0.201569
5,Linear Regression Core,linear_regression_core,stress_level_ord,1.274943
6,Linear Regression Full,linear_regression_full,age,0.016645
7,Linear Regression Full,linear_regression_full,gaming_hours,-2.041342
8,Linear Regression Full,linear_regression_full,study_hours,10.231738
9,Linear Regression Full,linear_regression_full,sleep_hours,5.669919


In [32]:
importance_rows = []

for spec in model_specs:
    fitted_model = spec['Model_object']

    if hasattr(fitted_model, 'feature_importances_'):
        feature_names = spec['X_train'].columns.tolist()
        for feature, importance in zip(feature_names, fitted_model.feature_importances_):
            importance_rows.append({
                'Model': spec['Model'],
                'Short_name': spec['Short_name'],
                'Feature': feature,
                'Importance': importance
            })

tree_feature_importance_raw = pd.DataFrame(importance_rows)

if len(tree_feature_importance_raw) > 0:
    tree_feature_importance_raw = tree_feature_importance_raw.sort_values(
        ['Model', 'Importance'],
        ascending=[True, False]
    )
    tree_feature_importance_raw.to_csv(OUTPUT_DIR + 'tables/tree_feature_importance_raw.csv', index=False)
    display(tree_feature_importance_raw.head(20))
else:
    print('No tree feature importances found.')

,Model,Short_name,Feature,Importance
40,Gradient Boosting,gradient_boosting,gaming_study_ratio,0.769964
25,Gradient Boosting,gradient_boosting,study_hours,0.104428
26,Gradient Boosting,gradient_boosting,sleep_hours,0.061821
27,Gradient Boosting,gradient_boosting,attendance,0.023703
41,Gradient Boosting,gradient_boosting,total_load,0.007730
24,Gradient Boosting,gradient_boosting,gaming_hours,0.007356
37,Gradient Boosting,gradient_boosting,gaming_hours_sq,0.007166
45,Gradient Boosting,gradient_boosting,gaming_over_2_5h,0.006006
43,Gradient Boosting,gradient_boosting,gaming_over_2h,0.004981
44,Gradient Boosting,gradient_boosting,gaming_over_1_5h,0.002969


## 10. Optional tuning hook

This section is included as a prepared extension, but it is not run by default.

Reason: Part 1 should avoid spending too much time on test-set optimization.  
If the team wants to tune the best nonlinear model later, the next teammate can enable this section and evaluate the tuned model responsibly.

In [33]:
RUN_OPTIONAL_TUNING = False

if RUN_OPTIONAL_TUNING:
    param_dist = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [2, 3, 4],
        'learning_rate': [0.03, 0.05, 0.08, 0.1],
        'subsample': [0.8, 0.9, 1.0]
    }

    gb = GradientBoostingRegressor(random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        estimator=gb,
        param_distributions=param_dist,
        n_iter=12,
        scoring='neg_root_mean_squared_error',
        cv=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search.fit(X_train_model[full_features], y_train)

    print('Best parameters:', search.best_params_)
    print('Best CV RMSE:', -search.best_score_)

    joblib.dump(search.best_estimator_, OUTPUT_DIR + 'models/tuned_gradient_boosting.joblib')

## Part 1 takeaway

Part 1 has completed the modelling-design and training responsibilities:

- loaded M3 processed data
- audited modelling data
- audited feature groups
- excluded a possible leakage-risk feature
- engineered EDA-informed threshold features
- built multiple design matrices
- trained baseline, linear, threshold, regularized, and nonlinear models
- computed raw CV and test metrics
- saved predictions and model artifacts
- exported coefficients and tree feature importances

The final evaluation, visualisations, and limitations are intentionally left for Part 2.

---

# Part 2 — To be completed by the next teammate

Recommended next steps:

1. Load `model_performance_raw.csv`.
2. Create a model comparison table/chart.
3. Select the best model using RMSE, MAE, and R².
4. Use `test_predictions.csv` for actual-vs-predicted plots.
5. Use `tree_feature_importance_raw.csv` for feature-importance plots.
6. Use saved models to create a gaming-hours threshold curve.
7. Conduct residual analysis and identify failure modes.
8. Write model limitations.

Suggested stakeholder-facing visualisations:

- Model comparison chart
- Actual vs predicted grades
- Feature importance chart
- Gaming-hours threshold curve
- Prediction error by gaming-hours group